In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3"
  },
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# FAS Inspector Hercules — YOLOv8 Training\n",
    "Train shape detector for cast-iron part verification.\n",
    "\n",
    "**Before running:**\n",
    "1. Runtime → Change runtime type → GPU\n",
    "2. Upload your images zip to Google Drive"
   ]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 1 — Install dependencies\n",
    "!pip install ultralytics -q"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 2 — Mount Google Drive\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 3 — Set your images path on Google Drive\n",
    "# Change this to where your images zip is located\n",
    "IMAGES_ZIP = '/content/drive/MyDrive/FAS_inspector/images.zip'\n",
    "\n",
    "import zipfile, os\n",
    "os.makedirs('/content/fas_data/images', exist_ok=True)\n",
    "with zipfile.ZipFile(IMAGES_ZIP, 'r') as z:\n",
    "    z.extractall('/content/fas_data/images')\n",
    "print('Images extracted')\n",
    "print(f'Total images: {len(os.listdir(\"/content/fas_data/images\"))}')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 4 — Upload label merge report\n",
    "from google.colab import files\n",
    "print('Upload your _label_merge_report JSON file')\n",
    "uploaded = files.upload()"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 5 — Prepare YOLO dataset from merge report\n",
    "import json, random, shutil\n",
    "from pathlib import Path\n",
    "\n",
    "CLASS_NAMES = ['SU61855', 'SU61856', 'SU65551RAW', 'SU66072RAW']\n",
    "CLASS_INDEX = {c: i for i, c in enumerate(CLASS_NAMES)}\n",
    "\n",
    "# Load merge report\n",
    "report_file = list(uploaded.keys())[0]\n",
    "with open(report_file) as f:\n",
    "    report = json.load(f)\n",
    "stem_to_variant = {item['stem']: item['variant'] for item in report['details']}\n",
    "print(f'Loaded {len(stem_to_variant)} labeled images from report')\n",
    "\n",
    "# Split train/val\n",
    "stems = list(stem_to_variant.keys())\n",
    "random.seed(42)\n",
    "random.shuffle(stems)\n",
    "n_val = max(1, int(len(stems) * 0.15))\n",
    "val_set = set(stems[:n_val])\n",
    "\n",
    "# Copy images and generate labels\n",
    "image_dir = Path('/content/fas_data/images')\n",
    "out_dir = Path('/content/fas_dataset')\n",
    "counts = {'train': 0, 'val': 0, 'skipped': 0}\n",
    "\n",
    "for stem, variant in stem_to_variant.items():\n",
    "    cls_idx = CLASS_INDEX.get(variant)\n",
    "    if cls_idx is None:\n",
    "        counts['skipped'] += 1\n",
    "        continue\n",
    "\n",
    "    # Find image file\n",
    "    img_path = None\n",
    "    for ext in ['.jpg', '.jpeg', '.png']:\n",
    "        p = image_dir / (stem + ext)\n",
    "        if p.exists():\n",
    "            img_path = p\n",
    "            break\n",
    "\n",
    "    if img_path is None:\n",
    "        counts['skipped'] += 1\n",
    "        continue\n",
    "\n",
    "    split = 'val' if stem in val_set else 'train'\n",
    "    img_dest = out_dir / 'images' / split / img_path.name\n",
    "    lbl_dest = out_dir / 'labels' / split / (stem + '.txt')\n",
    "\n",
    "    img_dest.parent.mkdir(parents=True, exist_ok=True)\n",
    "    lbl_dest.parent.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "    shutil.copy(img_path, img_dest)\n",
    "    lbl_dest.write_text(f'{cls_idx} 0.5 0.5 1.0 1.0\\n')\n",
    "    counts[split] += 1\n",
    "\n",
    "print(f'Train: {counts[\"train\"]}  Val: {counts[\"val\"]}  Skipped: {counts[\"skipped\"]}')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 6 — Write data.yaml\n",
    "yaml_content = f\"\"\"path: /content/fas_dataset\n",
    "train: images/train\n",
    "val: images/val\n",
    "\n",
    "nc: 4\n",
    "names:\n",
    "  0: SU61855\n",
    "  1: SU61856\n",
    "  2: SU65551RAW\n",
    "  3: SU66072RAW\n",
    "\"\"\"\n",
    "with open('/content/fas_dataset/data.yaml', 'w') as f:\n",
    "    f.write(yaml_content)\n",
    "print('data.yaml written')\n",
    "print(yaml_content)"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 7 — Train YOLOv8\n",
    "from ultralytics import YOLO\n",
    "\n",
    "model = YOLO('yolov8n.pt')\n",
    "\n",
    "results = model.train(\n",
    "    data='/content/fas_dataset/data.yaml',\n",
    "    epochs=80,\n",
    "    imgsz=640,\n",
    "    batch=16,\n",
    "    project='/content/runs',\n",
    "    name='fas_shape_v1',\n",
    "    exist_ok=True,\n",
    "    patience=20,\n",
    "    hsv_h=0.015,\n",
    "    hsv_s=0.3,\n",
    "    hsv_v=0.4,\n",
    "    degrees=10.0,\n",
    "    translate=0.1,\n",
    "    flipud=0.2,\n",
    "    fliplr=0.5,\n",
    ")\n",
    "print('Training complete!')"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 8 — Copy best weights and download\n",
    "import shutil\n",
    "from google.colab import files\n",
    "\n",
    "best = '/content/runs/fas_shape_v1/weights/best.pt'\n",
    "dest = '/content/shape_detector.pt'\n",
    "shutil.copy(best, dest)\n",
    "print('Downloading shape_detector.pt...')\n",
    "files.download(dest)"
   ],
   "execution_count": null,
   "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Step 9 — Verify model works\n",
    "from ultralytics import YOLO\n",
    "model = YOLO('/content/shape_detector.pt')\n",
    "print('Model classes:', model.names)\n",
    "print('Model loaded successfully — ready to use in FAS Inspector')"
   ],
   "execution_count": null,
   "outputs": []
  }
 ]
}